# Supervised Learning

## Learning Objectives
1. Implement logistic regression from scratch using gradient descent to understand the loss-gradient loop.
2. Build a sklearn pipeline comparing LR/SVC/DT/RF with cross-validation and OOM-safe batching.
3. Analyze feature importance across models to understand what each algorithm captures.
4. Investigate calibrated probabilities (Platt scaling) and data efficiency via learning curves.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Logistic Regression from Scratch

In [ ]:
# Binary logistic regression: p(y=1|x) = sigma(w^T x + b)
# Loss = binary cross-entropy; gradient = X^T (p - y) / n

def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


def bce_loss(y_true: np.ndarray, y_prob: np.ndarray, eps: float = 1e-9) -> float:
    """Binary cross-entropy loss."""
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))


class LogisticRegressionNumpy:
    """Logistic regression trained with mini-batch gradient descent."""

    def __init__(self, lr: float = 0.1, n_epochs: int = 100, batch_size: int = 32):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.w: np.ndarray | None = None
        self.b: float = 0.0

    def fit(self, X: np.ndarray, y: np.ndarray) -> list:
        """Train and return loss history."""
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        loss_hist = []
        for epoch in range(self.n_epochs):
            # Mini-batch gradient descent
            idx = np.random.permutation(n)
            for start in range(0, n, self.batch_size):
                batch_idx = idx[start:start + self.batch_size]
                Xb, yb = X[batch_idx], y[batch_idx]
                p = sigmoid(Xb @ self.w + self.b)
                error = p - yb
                self.w -= self.lr * (Xb.T @ error) / len(yb)
                self.b -= self.lr * error.mean()
            # Record full-batch loss every 10 epochs
            if epoch % 10 == 0:
                loss = bce_loss(y, sigmoid(X @ self.w + self.b))
                loss_hist.append((epoch, loss))
        return loss_hist

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return sigmoid(X @ self.w + self.b)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)


# Generate a linearly separable dataset
X_raw, y_raw = make_classification(
    n_samples=800, n_features=10, n_informative=5, random_state=42
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
split = 640

model_np = LogisticRegressionNumpy(lr=0.1, n_epochs=100)
loss_hist = model_np.fit(X_scaled[:split], y_raw[:split])
preds = model_np.predict(X_scaled[split:])
acc = (preds == y_raw[split:]).mean()

print(f"Logistic Regression (from scratch) val accuracy: {acc:.3f}")
print("Loss history (epoch, BCE loss):")
for ep, loss in loss_hist[:5]:
    print(f"  epoch {ep:3d}: loss = {loss:.4f}")

## Level 2: Sklearn Pipeline — LR/SVC/DT/RF with Cross-Validation

In [ ]:
# Compare four algorithms using proper cross-validation.
# Pipeline ensures scaling is done inside CV folds (no data leakage).

from sklearn.model_selection import StratifiedKFold

X_full, y_full = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, C=1.0))
    ]),
    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, probability=True))
    ]),
    'DecisionTree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=6, min_samples_leaf=10))
    ]),
    'RandomForest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':>20} | {'Mean CV Acc':>12} | {'Std':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 68)

cv_results = {}
for name, pipe in models.items():
    try:
        scores = cross_val_score(pipe, X_full, y_full, cv=cv, scoring='accuracy', n_jobs=1)
        cv_results[name] = scores
        print(f"{name:>20} | {scores.mean():12.3f} | {scores.std():8.3f} | "
              f"{scores.min():8.3f} | {scores.max():8.3f}")
    except MemoryError:
        print(f"{name:>20} | OOM — reduce n_samples or n_features")

## Real-World Example 1: Feature Importance Comparison

In [ ]:
# Different algorithms express feature importance differently:
# - LogisticRegression: |coefficient| (linear contribution)
# - RandomForest: mean decrease in impurity (non-linear, handles interactions)
# - SVC (linear): |w| from the decision hyperplane

from sklearn.inspection import permutation_importance

X_fi, y_fi = make_classification(
    n_samples=1000, n_features=15, n_informative=8, n_redundant=3,
    random_state=0, shuffle=False  # Features 0-7 are informative
)
scaler_fi = StandardScaler()
X_fi_s = scaler_fi.fit_transform(X_fi)
split_fi = 800

# Fit models
lr_fi = LogisticRegression(max_iter=500).fit(X_fi_s[:split_fi], y_fi[:split_fi])
rf_fi = RandomForestClassifier(n_estimators=100, random_state=42).fit(
    X_fi[:split_fi], y_fi[:split_fi]
)
svc_linear = SVC(kernel='linear', C=1.0).fit(X_fi_s[:split_fi], y_fi[:split_fi])

# Feature importances
lr_importance = np.abs(lr_fi.coef_[0])
rf_importance = rf_fi.feature_importances_
svc_importance = np.abs(svc_linear.coef_[0])

# Normalize for fair comparison
def normalize(a): return a / a.max()

print(f"{'Feature':>9} | {'LR |coef|':>10} | {'RF impurity':>12} | {'SVC |w|':>9} | {'Informative':>11}")
print("-" * 65)
for i in range(15):
    is_informative = "YES" if i < 8 else "no"
    print(f"  feat {i:2d}  | {normalize(lr_importance)[i]:10.3f} | "
          f"{normalize(rf_importance)[i]:12.3f} | {normalize(svc_importance)[i]:9.3f} | "
          f"{is_informative:>11}")

## Real-World Example 2: Calibrated Probabilities with Platt Scaling

In [ ]:
# Well-calibrated probabilities matter for:  
# risk scoring, threshold tuning, ensemble weighting.
# Platt scaling fits a sigmoid on the raw scores; isotonic regression is non-parametric.

from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split

X_cal, y_cal = make_classification(
    n_samples=3000, n_features=20, n_informative=10, random_state=7
)
X_tr, X_te, y_tr, y_te = train_test_split(X_cal, y_cal, test_size=0.3, random_state=0)

# SVC is known to produce poorly calibrated raw scores
svc_uncal = SVC(kernel='rbf', C=1.0, probability=False)
svc_uncal.fit(X_tr, y_tr)

svc_platt = CalibratedClassifierCV(
    SVC(kernel='rbf', C=1.0, probability=False), method='sigmoid', cv=5
)
svc_platt.fit(X_tr, y_tr)

svc_isotonic = CalibratedClassifierCV(
    SVC(kernel='rbf', C=1.0, probability=False), method='isotonic', cv=5
)
svc_isotonic.fit(X_tr, y_tr)

# Use decision_function scores for uncalibrated SVC
raw_scores = svc_uncal.decision_function(X_te)
# Manually map to [0,1] with min-max for illustration
uncal_probs = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
platt_probs = svc_platt.predict_proba(X_te)[:, 1]
iso_probs = svc_isotonic.predict_proba(X_te)[:, 1]

print(f"{'Method':>12} | {'Brier Score':>12} | {'Lower is better':>15}")
print("-" * 45)
for name, probs in [('Uncalibrated', uncal_probs), ('Platt', platt_probs), ('Isotonic', iso_probs)]:
    bs = brier_score_loss(y_te, probs)
    print(f"{name:>12} | {bs:12.4f}")

print("\nBrier score = mean squared error of predicted probabilities.")
print("Platt scaling: fast, assumes sigmoid mapping. Good for SVC, NB.")
print("Isotonic: flexible, requires more data. Better for tree ensembles.")

## Real-World Example 3: Learning Curves — Data Efficiency

In [ ]:
# Learning curves show how much data each algorithm needs to converge.
# Helpful for: deciding whether to collect more data vs tune the model.
# High train/low val = overfitting; both low = underfitting; both high = good fit.

X_lc, y_lc = make_classification(
    n_samples=5000, n_features=20, n_informative=10, random_state=3
)

estimators = {
    'LR': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=500))]),
    'RF': RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42),
}

train_sizes_abs = np.array([100, 250, 500, 1000, 2000, 3500])

print(f"{'Model':>5} | {'N Train':>9} | {'Train Acc':>10} | {'Val Acc':>9}")
print("-" * 45)
for name, est in estimators.items():
    train_sizes, train_scores, val_scores = learning_curve(
        est, X_lc, y_lc,
        train_sizes=train_sizes_abs,
        cv=3, scoring='accuracy', n_jobs=1
    )
    for n, tr, vl in zip(train_sizes, train_scores.mean(1), val_scores.mean(1)):
        print(f"{name:>5} | {n:9d} | {tr:10.3f} | {vl:9.3f}")
    print()

print("Guidance:")
print("  LR: val accuracy plateaus early — more data yields diminishing returns.")
print("  RF: continues to benefit from more data; high train acc = some overfitting.")
print("  If val acc keeps rising with more data: collect more. If flat: tune model.")